# Ensemble — stack boosters (+external) with RealMLP

Probability-level ensembler. Loads the OOF / test probabilities saved by:
- `modeling_note.ipynb` -> `oof_lgb/xgb/cb.npy`, `test_lgb/xgb/cb.npy` (boosters trained **with** external SDSS17)
- `realmlp_note.ipynb`  -> `oof_mlp.npy`, `test_mlp.npy` (RealMLP, also **with** external)

All share the same `StratifiedKFold(5, shuffle=True, random_state=42)` over the real rows, so
the OOF arrays align row-by-row. We:
1. report each model's OOF balanced accuracy;
2. compare a **boosters-only** stack vs **boosters + RealMLP** stack (to quantify the MLP gain);
3. fit a logistic-regression **meta-model** on the stacked OOF (honest via `cross_val_predict`);
4. tune **per-class multipliers** for balanced accuracy and write `submission.csv`.

On Kaggle: add both source notebooks' outputs as inputs to this notebook.

## Load saved OOF / test arrays

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

SEED, FOLDS, NC = 42, 5, 3
CLASSES = ['GALAXY', 'QSO', 'STAR']
ID, TARGET = 'id', 'class'

def find(name):
    hits = sorted(Path('/kaggle/input').rglob(name)) if Path('/kaggle/input').exists() else []
    if hits:
        return hits[0]
    local = Path('../data_processed') / name
    if local.exists():
        return local
    raise FileNotFoundError(f'{name} not found (run modeling_note / realmlp_note first, '
                            f'and on Kaggle add their outputs as inputs)')

def load(name):
    p = find(name); print(f'  {name:14s} <- {p}')
    return np.load(p, allow_pickle=True)

print('loading boosters:')
oof_lgb, oof_xgb, oof_cb = load('oof_lgb.npy'), load('oof_xgb.npy'), load('oof_cb.npy')
test_lgb, test_xgb, test_cb = load('test_lgb.npy'), load('test_xgb.npy'), load('test_cb.npy')
print('loading realmlp:')
oof_mlp, test_mlp = load('oof_mlp.npy'), load('test_mlp.npy')
y = load('oof_y.npy').astype(int)
test_id = load('test_id.npy')

# sanity: every OOF must cover the same rows in the same order
n = len(y)
for nm, a in [('lgb',oof_lgb),('xgb',oof_xgb),('cb',oof_cb),('mlp',oof_mlp)]:
    assert a.shape == (n, NC), f'{nm} OOF shape {a.shape} != {(n, NC)}'
nt = len(test_id)
for nm, a in [('lgb',test_lgb),('xgb',test_xgb),('cb',test_cb),('mlp',test_mlp)]:
    assert a.shape == (nt, NC), f'{nm} test shape {a.shape} != {(nt, NC)}'
print('rows: train', n, '| test', nt, '| all arrays aligned')

## Individual model OOF scores

In [ ]:
models = {'lgb': (oof_lgb, test_lgb), 'xgb': (oof_xgb, test_xgb),
          'cb': (oof_cb, test_cb), 'mlp': (oof_mlp, test_mlp)}
for k, (o, _) in models.items():
    print(f'{k:4s} OOF BA: {balanced_accuracy_score(y, o.argmax(1)):.5f}')

## Stacking: boosters-only vs boosters + RealMLP

A multinomial logistic-regression meta-model on the stacked OOF probabilities. We score it
honestly with `cross_val_predict`, then refit on the full OOF to transform the test set.

In [ ]:
def stack_oof(keys):  return np.hstack([models[k][0] for k in keys]).astype('float64')
def stack_test(keys): return np.hstack([models[k][1] for k in keys]).astype('float64')

skf = StratifiedKFold(FOLDS, shuffle=True, random_state=SEED)

def fit_meta(keys, label):
    S = stack_oof(keys)
    meta = LogisticRegression(penalty='l2', C=1.0, max_iter=2000,
                              class_weight='balanced', n_jobs=-1)
    oof = cross_val_predict(meta, S, y, cv=skf, method='predict_proba', n_jobs=-1)
    ba = balanced_accuracy_score(y, oof.argmax(1))
    print(f'{label:28s} meta-OOF BA: {ba:.5f}')
    meta.fit(S, y)
    test = meta.predict_proba(stack_test(keys))
    return oof, test, ba

oof_gbm_only, test_gbm_only, ba_gbm = fit_meta(['lgb','xgb','cb'], 'boosters-only stack')
oof_all, test_all, ba_all = fit_meta(['lgb','xgb','cb','mlp'], 'boosters + RealMLP stack')
print(f'\nRealMLP contribution to stack: {ba_all - ba_gbm:+.5f}')

# use the stronger stack downstream
blend_oof, blend_test = (oof_all, test_all) if ba_all >= ba_gbm else (oof_gbm_only, test_gbm_only)

## Per-class weight tuning + final diagnostics

Coordinate search of per-class multipliers on the meta-OOF (balanced accuracy is maximized by
trading a little GALAXY recall for STAR/QSO at the confusing boundary).

In [ ]:
def ba_w(proba, w): return balanced_accuracy_score(y, (proba * w).argmax(1))

w = np.ones(NC); base = ba_w(blend_oof, w)
for _ in range(40):
    improved = False
    for c in range(NC):
        for mult in np.linspace(0.5, 2.0, 31):
            wt = w.copy(); wt[c] = mult
            s = ba_w(blend_oof, wt)
            if s > base + 1e-6:
                base, w, improved = s, wt, True
    if not improved:
        break
w = w / w.mean()
print('class multipliers:', dict(zip(CLASSES, w.round(3))))
print('stack BA argmax  : %.5f' % ba_w(blend_oof, np.ones(NC)))
print('stack BA weighted: %.5f' % base)
print()
pred = (blend_oof * w).argmax(1)
print(classification_report(y, pred, target_names=CLASSES, digits=4))
print('confusion matrix (rows=true):')
print(pd.DataFrame(confusion_matrix(y, pred), index=CLASSES, columns=CLASSES))

## Submission

In [ ]:
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('../data_processed')
int_to_class = {i: c for i, c in enumerate(CLASSES)}
test_pred = (blend_test * w).argmax(1)
sub = pd.DataFrame({ID: test_id, TARGET: [int_to_class[i] for i in test_pred]})
sub.to_csv(OUT / 'submission.csv', index=False)
print('Saved', (OUT / 'submission.csv').resolve(), sub.shape)
print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()